In [5]:
import os
import json
import pandas as pd

# ── 載入所有推文 ──────────────────────────────────────
def load_pheme(base_path):
    records = []
    
    for event in os.listdir(base_path):
        event_path = os.path.join(base_path, event)
        if not os.path.isdir(event_path):
            continue
            
        for label in ["rumours", "non-rumours"]:
            label_path = os.path.join(event_path, label)
            if not os.path.exists(label_path):
                continue
                
            for thread_id in os.listdir(label_path):
                thread_path = os.path.join(label_path, thread_id)
                
                # 讀原始推文
                source_path = os.path.join(thread_path, "source-tweets")
                if os.path.exists(source_path):
                    for fname in os.listdir(source_path):
                        if fname.startswith("._"):
                            continue
                        fpath = os.path.join(source_path, fname)
                        if os.path.getsize(fpath) == 0:
                            continue
                        try:
                            with open(os.path.join(source_path, fname), encoding="utf-8", errors="ignore") as f:
                                tweet = json.load(f)
                                records.append({
                                    "event"     : event,
                                    "thread_id" : thread_id,
                                    "tweet_id"  : tweet.get("id_str"),
                                    "text"      : tweet.get("text"),
                                    "user"      : tweet.get("user", {}).get("screen_name"),
                                    "created_at": tweet.get("created_at"),
                                    "retweet_count": tweet.get("retweet_count"),
                                    "is_rumour" : label == "rumours",
                                    "type"      : "source"
                                })
                        except json.JSONDecodeError:
                            print(f"無法解析 JSON：{fpath}")
                            continue
    return pd.DataFrame(records)

# ── 載入資料 ──────────────────────────────────────────
print(os.getcwd())
df = load_pheme("data/raw/pheme")
print(f"總推文數：{len(df)}")
print(f"欄位：{df.columns.tolist()}")
df.head()

/app
無法解析 JSON：data/raw/pheme/sydneysiege-all-rnr-threads/rumours/544490782183657474/source-tweets/.DS_Store
總推文數：6425
欄位：['event', 'thread_id', 'tweet_id', 'text', 'user', 'created_at', 'retweet_count', 'is_rumour', 'type']


,event,thread_id,tweet_id,text,user,created_at,retweet_count,is_rumour,type
0,prince-toronto-all-rnr-threads,529694805010685952,529694805010685952,$10 to see Prince tonight in Toronto? I'd be a...,ajsharma95,Tue Nov 04 18:00:57 +0000 2014,3,True,source
1,prince-toronto-all-rnr-threads,529621484130828290,529621484130828290,"Just noticed ""HALL of desire"" in that @drfunke...",tcote,Tue Nov 04 13:09:36 +0000 2014,2,True,source
2,prince-toronto-all-rnr-threads,529723362894176256,529723362894176256,Prince fans lined up outside Massey Hall say t...,stephaniesmyth,Tue Nov 04 19:54:26 +0000 2014,3,True,source
3,prince-toronto-all-rnr-threads,529670383629520896,529670383629520896,Surprise Prince show at Massey Hall tonight? h...,TheTorontoSun,Tue Nov 04 16:23:54 +0000 2014,6,True,source
4,prince-toronto-all-rnr-threads,529749285198655488,529749285198655488,WOW. It seems as though #Prince will not be pl...,YYZMag,Tue Nov 04 21:37:26 +0000 2014,2,True,source


In [11]:
#print(df.head())

print(df.groupby("event").size())

print(df["is_rumour"].value_counts())

print(df.isna().sum())

event
charliehebdo-all-rnr-threads         2079
ebola-essien-all-rnr-threads           14
ferguson-all-rnr-threads             1143
germanwings-crash-all-rnr-threads     469
gurlitt-all-rnr-threads               138
ottawashooting-all-rnr-threads        890
prince-toronto-all-rnr-threads        233
putinmissing-all-rnr-threads          238
sydneysiege-all-rnr-threads          1221
dtype: int64
is_rumour
False    4023
True     2402
Name: count, dtype: int64
event            0
thread_id        0
tweet_id         0
text             0
user             0
created_at       0
retweet_count    0
is_rumour        0
type             0
dtype: int64
